# Ring Resonator (Cascaded Hybrid) -- L_C x L_SIDE Resonance Sweep

Sweeps `L_C` (coupling straight length) and `L_SIDE` (side straight length)
over a grid and heatmaps each point's FSR, FWHM, and extinction ratio.

**No new simulations run here.** `run_left_fdtd()`/`run_bend_fdtd()`'s cache
key stopped depending on `L_C` in the 2026-08-04 fix, and
`solve_body_supermodes()`/`solve_neff()` never depended on `L_C`/`L_SIDE` at
all -- both only enter the final analytic assembly (`BODY_LENGTH`,
`L_OUTER`), never the cached FDTD/mode-solve building blocks. So this
notebook loads those building blocks once (`cache_only=True` -- a missing
cache entry raises `CacheMissError` rather than launching a simulation) and
re-derives the transmission spectrum for every grid point purely with numpy,
reusing the exact assembly logic from `ring_resonator_hybrid_cascaded.py`
(ported in `ring_resonator_sweep_helpers.py`, same directory).

Per the validated route's own analysis conventions
(`validation_markdown/resolved/ring_resonator_validation.md`): absolute
resonance wavelength is out of scope (round-trip phase error is amplified
too much by this device's finesse to validate an absolute comb position) --
this sweep only reports FSR/FWHM/extinction, which are relative/shape
quantities, not absolute positions.

Grid points where `BEND_STRAIGHT_HEADROOM < 0` (the bend's own stubs would
cover more straight than the device physically has -- see
`ring_resonator_hybrid_cascaded.py`'s own guard) or where fewer than 2
resonance dips are detected (e.g. an under-coupled operating point where the
resonance extinction never crosses the detection threshold) are masked
invalid rather than plotted.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import ring_resonator_sweep_helpers as sweep


In [2]:
import os

# Same convention as every other example: output lives outside the repo,
# in example_output/, one folder per example (this one's own subfolder so
# it doesn't collide with ring_resonator_hybrid_cascaded's own outputs).
_HERE = os.getcwd()
output_dir = os.path.abspath(os.path.join(
    _HERE, "..", "..", "..", "..", "example_output", "ring_resonator",
    "ring_resonator_hybrid_cascaded_sweep"))
os.makedirs(output_dir, exist_ok=True)


In [3]:
# One-time load of every cached FDTD/mode-solve building block. cache_only=True
# throughout ring_resonator_sweep_helpers.py: this raises CacheMissError instead
# of ever launching a new simulation.
blocks = sweep.load_cached_blocks()


Racetrack perimeter: 43.42 um  ->  predicted FSR ~ 12.87 nm (~7.8 resonances across 1.5-1.6 um)
Section previews saved -> /Users/jhiesener/Documents/Research/duty/example_output/ring_resonator/ring_resonator_hybrid_cascaded/sections/ (fdtd_left_section, fdtd_standalone_bend, eme_body_cross_section, eme_strip_cross_section)
Assembly diagram saved -> /Users/jhiesener/Documents/Research/duty/example_output/ring_resonator/ring_resonator_hybrid_cascaded/geometry_assembly.png

Round-trip ledger — assigned length vs physical span between reference planes
  span                                        modelled by                             assigned  physical  mismatch
  ----------------------------------------------------------------------------------------------------------------
  left section: ring port -> interface        M_left (FDTD)                           measured    9.1040         —
  body: interface -> interface                P_body (supermode phase)                  2.5000    2.5

In [4]:
# L_C / L_SIDE grid. Deliberately wide so the heatmap also shows the boundary
# of the valid region (BEND_STRAIGHT_HEADROOM >= 0 and >=2 resonances found),
# not just a comfortable interior.
N_GRID = 101
L_C_grid = np.linspace(2.0, 6.0, N_GRID)
L_SIDE_grid = np.linspace(1.0, 60.0, N_GRID)

fsr = np.full((N_GRID, N_GRID), np.nan)
fwhm = np.full((N_GRID, N_GRID), np.nan)
extinction_db = np.full((N_GRID, N_GRID), np.nan)
n_invalid = 0

for i, L_C in enumerate(L_C_grid):
    for j, L_SIDE in enumerate(L_SIDE_grid):
        r = sweep.assemble(blocks, L_C=L_C, L_SIDE=L_SIDE)
        if r["valid"]:
            fsr[i, j] = r["fsr"] * 1000        # nm
            fwhm[i, j] = r["fwhm"] * 1000       # nm
            extinction_db[i, j] = r["extinction_db"]
        else:
            n_invalid += 1

print(f"{N_GRID * N_GRID - n_invalid}/{N_GRID * N_GRID} grid points valid "
      f"({n_invalid} masked -- headroom<0 or resonance not detected)")


10130/10201 grid points valid (71 masked -- headroom<0 or resonance not detected)


In [5]:
_plots = [
    (fsr, "FSR", "FSR (nm)", "fsr.png"),
    (fwhm, "FWHM", "FWHM (nm)", "fwhm.png"),
    (extinction_db, "Extinction ratio", "Extinction (dB)", "extinction.png"),
]

for data, title, cbar_label, filename in _plots:
    fig, ax = plt.subplots(figsize=(6, 4.5))
    im = ax.pcolormesh(L_SIDE_grid, L_C_grid, data, shading="nearest", cmap="viridis")
    ax.set_xlabel("L_SIDE (um)")
    ax.set_ylabel("L_C (um)")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label=cbar_label)
    plt.tight_layout()
    fig.savefig(os.path.join(output_dir, filename), dpi=150)
    plt.show()

print(f"Plots saved -> {output_dir}/{{fsr,fwhm,extinction}}.png")


Plots saved -> /Users/jhiesener/Documents/Research/duty/example_output/ring_resonator/ring_resonator_hybrid_cascaded_sweep/{fsr,fwhm,extinction}.png


/var/folders/sr/q5h5rlhd3wj7zf4dl9fvm_1r0000gn/T/ipykernel_22361/2155035858.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
